# Sub-Region Force Generalisation — Validation Notebook

**Concept**: single simulation, same N³ particles.
- **Coarse force** (F_coarse): PM on `mesh_lr`
- **Fine force** (F_fine): PM on `mesh_hr` — more accurate, same particles
- **Target**: `ΔF = F_fine − F_coarse`
- **Training region**: a Lagrangian spatial sub-cube (fraction of the simulation)
- **Test region**: the rest of the simulation

**Key question**: does the model trained on a sub-region generalise to the full simulation?

| Section | Question |
|---------|----------|
| 0 | Config & data loading |
| 1 | Force pair sanity — |ΔF| distribution, signal strength |
| 2 | Sub-region split — visualise train vs test particles |
| 3 | Feature–force correlation (train region vs test region) |
| 4 | Model inference — train R vs test R |
| 5 | Spatial generalisation maps |
| 6 | Across snapshots |

In [ ]:
%load_ext autoreload
%autoreload 2

import yaml, pickle
from pathlib import Path
from types import SimpleNamespace
from functools import partial

import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr

import sys
REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT / "pm2nbody"))

from train_lag_force      import compute_force_pair, snapshot_features
from train_subregion_forceres import load_single_snapshot, make_patch_split
from jaxpm.lagrangian     import get_axis_neighbor_indices, make_lagrangian_corrector

print("imports OK  |  JAX:", jax.devices())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIG — edit here
# ═══════════════════════════════════════════════════════════════════
CONFIG_PATH = REPO_ROOT / "configs/subregion_forceres.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

data_cfg  = SimpleNamespace(**cfg["data"])
model_cfg = SimpleNamespace(**cfg["model"])
train_cfg = SimpleNamespace(**cfg["training"])

DATA_DIR      = Path(data_cfg.data_dir)
N_PART        = int(data_cfg.n_part)
MESH_LR       = int(data_cfg.mesh_lr)
MESH_HR       = int(data_cfg.mesh_hr)
BOX_SIZE      = float(data_cfg.box_size)
SIM_TRAIN     = int(data_cfg.sim_id_train)
SIM_VAL       = int(getattr(data_cfg, "sim_id_val", 1))
SNAP_TRAIN    = int(data_cfg.snap_train)
SNAPS_VAL     = list(data_cfg.snaps_val)
TRAIN_PATCH_N = int(train_cfg.train_patch_n)
PATCH_SEED    = int(getattr(train_cfg, "patch_seed", 42))
USE_STRAIN    = bool(model_cfg.use_strain)
USE_INVS      = bool(model_cfg.use_invariants)
USE_VEL       = bool(model_cfg.use_velocity)
CKPT_DIR      = REPO_ROOT / "runs/subregion_forceres"

train_frac = (TRAIN_PATCH_N / N_PART) ** 3
print(f"n_part={N_PART}  mesh_lr={MESH_LR}  mesh_hr={MESH_HR}  box={BOX_SIZE} Mpc/h")
print(f"train_patch_n={TRAIN_PATCH_N}  → {TRAIN_PATCH_N**3:,} train / "
      f"{N_PART**3 - TRAIN_PATCH_N**3:,} test  ({train_frac:.1%} of sim)")

# ── Shared structures ────────────────────────────────────────────────────────
neighbor_idx = get_axis_neighbor_indices(N_PART)
train_idx, test_idx, _ = make_patch_split(N_PART, TRAIN_PATCH_N, seed=PATCH_SEED)

# ── Load snapshot ─────────────────────────────────────────────────────────────
pos, vel, a = load_single_snapshot(DATA_DIR, SIM_TRAIN, SNAP_TRAIN, N_PART, BOX_SIZE)
print(f"\na={a:.4f}  pos range [{pos.min():.1f}, {pos.max():.1f}] mesh units")

# ── Force pair (same particles, two mesh sizes) ───────────────────────────────
_fp = jax.jit(partial(compute_force_pair, mesh_lr=MESH_LR, mesh_hr=MESH_HR))
f_coarse, f_fine, delta_f = _fp(pos)

f_coarse_np = np.asarray(jax.device_get(f_coarse))
f_fine_np   = np.asarray(jax.device_get(f_fine))
delta_f_np  = np.asarray(jax.device_get(delta_f))
pos_np      = np.asarray(jax.device_get(pos))

df_mag  = np.sqrt(np.sum(delta_f_np**2, axis=-1))
fc_mag  = np.sqrt(np.sum(f_coarse_np**2, axis=-1))
ff_mag  = np.sqrt(np.sum(f_fine_np**2,  axis=-1))

print(f"\n|F_coarse| mean = {fc_mag.mean():.4e}")
print(f"|F_fine|   mean = {ff_mag.mean():.4e}")
print(f"|ΔF|       mean = {df_mag.mean():.4e}  ({df_mag.mean()/ff_mag.mean():.1%} of F_fine)")

## Section 1 — Force pair sanity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ss = np.random.default_rng(0).choice(len(df_mag), min(50_000, len(df_mag)), replace=False)

# ΔF distribution (x component)
axes[0].hist(delta_f_np[:, 0], bins=200, alpha=0.7, density=True, color="C0")
axes[0].set_xlabel("ΔF_x [mesh units]"); axes[0].set_title("ΔF_x distribution")

# |ΔF| vs |F_fine|
axes[1].hexbin(ff_mag[ss], df_mag[ss], gridsize=70, norm=LogNorm(), mincnt=1, cmap="plasma")
axes[1].set_xlabel("|F_fine|"); axes[1].set_ylabel("|ΔF|")
axes[1].set_title(f"|ΔF| vs |F_fine|  a={a:.3f}")

# |F_coarse| vs |F_fine|
r_ff, _ = pearsonr(fc_mag[ss], ff_mag[ss])
axes[2].hexbin(ff_mag[ss], fc_mag[ss], gridsize=70, norm=LogNorm(), mincnt=1, cmap="viridis")
axes[2].plot([0, ff_mag.max()], [0, ff_mag.max()], "r--", lw=1, label="identity")
axes[2].set_xlabel("|F_fine|"); axes[2].set_ylabel("|F_coarse|")
axes[2].set_title(f"R={r_ff:.4f}  (coarse vs fine)")
axes[2].legend()

plt.suptitle(f"Force pair  (mesh_lr={MESH_LR} vs mesh_hr={MESH_HR})  a={a:.3f}", fontsize=11)
plt.tight_layout(); plt.show()

# Slab maps: F_coarse, F_fine, ΔF
slab_mask = np.abs(pos_np[:, 2] % N_PART - N_PART//2) < max(1, N_PART//20)
px, py    = pos_np[slab_mask, 0], pos_np[slab_mask, 1]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmax = np.percentile(ff_mag, 97)
for ax, (mag, label, cmap) in zip(axes, [
    (fc_mag[slab_mask], f"|F_coarse|  (mesh {MESH_LR}³)", "Blues"),
    (ff_mag[slab_mask], f"|F_fine|    (mesh {MESH_HR}³)", "Reds"),
    (df_mag[slab_mask], "|ΔF|",                           "Oranges"),
]):
    hb = ax.hexbin(px, py, C=mag, gridsize=60, cmap=cmap,
                   reduce_C_function=np.mean, vmin=0, vmax=vmax)
    plt.colorbar(hb, ax=ax); ax.set_title(label)
plt.suptitle(f"Force slab projection  a={a:.3f}", fontsize=11)
plt.tight_layout(); plt.show()

## Section 2 — Sub-region split: train vs test

In [ ]:
# ── Lagrangian positions ──────────────────────────────────────────────────
q = np.stack(np.meshgrid(*[np.arange(N_PART)]*3, indexing='ij'), axis=-1).reshape(-1, 3)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Lagrangian view: train vs test
ax = axes[0]
ax.scatter(q[test_idx,  0], q[test_idx,  1], s=0.3, c="steelblue", alpha=0.3, label="test")
ax.scatter(q[train_idx, 0], q[train_idx, 1], s=0.5, c="tomato",    alpha=0.8, label="train")
ax.set_title(f"Lagrangian split  (train={TRAIN_PATCH_N}³ patch, {train_frac:.1%})")
ax.set_xlabel("ix"); ax.set_ylabel("iy"); ax.legend(markerscale=6)

# Eulerian view: |ΔF| coloured by train/test
ax = axes[1]
color = np.where(np.isin(np.arange(N_PART**3), train_idx), "tomato", "steelblue")
for region, idx, label, col in [
    ("test",  test_idx,  "test",  "steelblue"),
    ("train", train_idx, "train", "tomato"),
]:
    ax.scatter(pos_np[idx, 0], pos_np[idx, 1],
               s=0.3, c=col, alpha=0.4, label=label)
ax.set_title("Eulerian view (train patch red, test blue)")
ax.set_xlabel("x [mesh units]"); ax.legend(markerscale=6)

plt.tight_layout(); plt.show()

# ── ΔF distribution comparison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, np.percentile(df_mag, 99), 80)
axes[0].hist(df_mag[train_idx], bins=bins, alpha=0.6, density=True, color="tomato",    label="train")
axes[0].hist(df_mag[test_idx],  bins=bins, alpha=0.6, density=True, color="steelblue", label="test")
axes[0].set_xlabel("|ΔF|"); axes[0].set_ylabel("density")
axes[0].set_title("|ΔF| distribution: train vs test region")
axes[0].legend()

# SC fraction
feats_all, det_D_all = snapshot_features(pos, neighbor_idx, N_PART, USE_STRAIN, USE_INVS)
det_D_np = np.asarray(jax.device_get(det_D_all))
axes[1].bar(["train", "test"], [(det_D_np[train_idx]<0).mean()*100,
                                 (det_D_np[test_idx]<0).mean()*100],
            color=["tomato","steelblue"], alpha=0.8)
axes[1].set_ylabel("Shell-crossing %")
axes[1].set_title("SC fraction per region")

plt.suptitle("Train vs test region properties", fontsize=11)
plt.tight_layout(); plt.show()

print(f"Train: mean|ΔF|={df_mag[train_idx].mean():.4e}  SC={((det_D_np[train_idx]<0).mean()*100):.1f}%")
print(f"Test:  mean|ΔF|={df_mag[test_idx].mean():.4e}  SC={((det_D_np[test_idx]<0).mean()*100):.1f}%")

## Section 3 — Feature–force correlation: train vs test

In [ ]:
# If features correlate equally with |ΔF| in both regions, the model has a
# chance to generalise. If train/test correlations differ a lot, expect poor generalisation.
feats_np  = np.asarray(jax.device_get(feats_all))
feat_dim  = feats_np.shape[1]
strain_mag = np.sqrt(np.sum(feats_np[:, :9]**2, axis=-1))

def region_r(idx, feat_col):
    return pearsonr(feats_np[idx, feat_col], df_mag[idx])[0]

r_train = [region_r(train_idx, fi) for fi in range(feat_dim)]
r_test  = [region_r(test_idx,  fi) for fi in range(feat_dim)]
r_full  = [pearsonr(feats_np[:, fi], df_mag)[0] for fi in range(feat_dim)]

feat_names = [f"E_{ab}" for ab in ["xx","xy","xz","yx","yy","yz","zx","zy","zz"]]
feat_names += [f"f{i}" for i in range(feat_dim - len(feat_names))]

fig, ax = plt.subplots(figsize=(max(10, feat_dim * 0.8), 4))
x = np.arange(feat_dim)
ax.bar(x - 0.25, r_train, 0.25, color="tomato",    alpha=0.8, label="train region")
ax.bar(x,        r_full,  0.25, color="gray",       alpha=0.6, label="full sim")
ax.bar(x + 0.25, r_test,  0.25, color="steelblue",  alpha=0.8, label="test region")
ax.set_xticks(x); ax.set_xticklabels(feat_names[:feat_dim], rotation=45, ha="right")
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("Pearson R  with  |ΔF|")
ax.set_title("Feature–force correlation: train vs test region  (similar = good generalisation)")
ax.legend(); plt.tight_layout(); plt.show()

# Summary: how aligned are train and test correlations?
r_agreement = pearsonr(r_train, r_test)[0]
print(f"Correlation between train_R and test_R across features: {r_agreement:.4f}")
print(f"  > 0.9 → features behave similarly in both regions → model should generalise")
print(f"  < 0.7 → features have different structure in train vs test → harder to generalise")

## Section 4 — Model inference: train R vs test R (generalisation gap)

The key metric: `train_R - test_R`. If the gap is small, the model trained
on the sub-region generalises to the rest of the simulation.

In [ ]:
PARAMS = None
lag_model = make_lagrangian_corrector(
    hidden_dim=int(model_cfg.hidden_dim),
    n_layers=int(model_cfg.n_layers), output_dim=3
)

# Load checkpoint (saved by train_subregion_forceres.py)
for rd in sorted(CKPT_DIR.glob("*/"), key=lambda p: p.stat().st_mtime, reverse=True):
    for fname in ["best_params.pkl", "final_params.pkl"]:
        pkl = rd / fname
        if pkl.exists():
            with open(pkl, "rb") as fh:
                PARAMS = hk.data_structures.to_immutable_dict(pickle.load(fh))
            print(f"Loaded: {pkl}"); break
    if PARAMS is not None: break

if PARAMS is None: print("No checkpoint — run train_subregion_forceres.py first")

if PARAMS is not None:
    vel_feat = vel if USE_VEL else jnp.zeros_like(vel)

    # Predict for ALL particles
    pred_all = np.asarray(jax.device_get(
        jax.jit(lag_model.apply)(PARAMS, feats_all, vel_feat, jnp.array(a))
    ))

    # ── Metrics per region ────────────────────────────────────────────────
    def region_metrics(idx, label):
        p, t = pred_all[idx], delta_f_np[idx]
        mse  = float(np.mean((p - t)**2))
        r_xyz = [pearsonr(p[:, c], t[:, c])[0] for c in range(3)]
        r_mean = float(np.mean(r_xyz))
        mse_baseline = float(np.mean(t**2))   # MSE of predicting zero
        return dict(label=label, mse=mse, r=r_mean,
                    r_xyz=r_xyz, mse_improvement=1-mse/mse_baseline)

    tm = region_metrics(train_idx, "train")
    vm = region_metrics(test_idx,  "test")
    fm = region_metrics(np.arange(len(pred_all)), "full")

    print(f"\n{'Region':<8} | {'R_mean':>8} | {'MSE_improv%':>12} | {'R_x':>6} {'R_y':>6} {'R_z':>6}")
    print("-" * 60)
    for m in [tm, vm, fm]:
        print(f"{m['label']:<8} | {m['r']:>8.4f} | {m['mse_improvement']*100:>11.1f}% | "
              f"{m['r_xyz'][0]:>6.3f} {m['r_xyz'][1]:>6.3f} {m['r_xyz'][2]:>6.3f}")

    gap = tm['r'] - vm['r']
    print(f"\nGeneralisation gap  (train R - test R) = {gap:+.4f}")
    if   abs(gap) < 0.05: print("  ✓ EXCELLENT — model generalises well across the simulation")
    elif abs(gap) < 0.15: print("  ✓ GOOD — modest generalisation gap")
    else:                 print("  ✗ LARGE GAP — model overfits to sub-region")

    # ── Scatter: train vs test ────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for row, (idx, label, color) in enumerate([(train_idx, "Train", "tomato"),
                                                (test_idx,  "Test",  "steelblue")]):
        ss2 = np.random.default_rng(row).choice(len(idx), min(30_000, len(idx)), replace=False)
        idx_ss = idx[ss2]
        for ci, comp in enumerate(["x", "y", "z"]):
            tgt  = delta_f_np[idx_ss, ci]
            pred = pred_all[idx_ss, ci]
            lim  = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15
            ax   = axes[row, ci]
            h    = ax.hexbin(tgt, pred, gridsize=60, cmap="Blues" if row==0 else "Greens",
                             norm=LogNorm(), mincnt=1, extent=[-lim, lim, -lim, lim])
            plt.colorbar(h, ax=ax)
            ax.plot([-lim, lim], [-lim, lim], "r--", lw=1)
            r_here = pearsonr(tgt, pred)[0]
            ax.set_title(f"{label} {comp}  R={r_here:.4f}")
            ax.set_xlabel(f"ΔF_{comp} target"); ax.set_ylabel("predicted")
    plt.suptitle(f"Prediction scatter: train region vs test region  a={a:.3f}", fontsize=11)
    plt.tight_layout(); plt.show()

## Section 5 — Spatial generalisation maps

Show the prediction error spatially. If the model generalises, the error map should
look similar inside and outside the training region.

In [ ]:
if PARAMS is not None:
    err_all = np.sqrt(np.sum((pred_all - delta_f_np)**2, axis=-1))   # per-particle error

    # Corrected force
    f_corrected = f_coarse_np + pred_all

    # MSE improvement globally
    mse_baseline  = float(np.mean((f_coarse_np - f_fine_np)**2))
    mse_corrected = float(np.mean((f_corrected  - f_fine_np)**2))
    improv = 1 - mse_corrected / mse_baseline
    print(f"MSE improvement (full sim):  {improv:.1%}")
    print(f"  train region: {1 - np.mean((f_corrected[train_idx] - f_fine_np[train_idx])**2) / np.mean((f_coarse_np[train_idx] - f_fine_np[train_idx])**2):.1%}")
    print(f"  test  region: {1 - np.mean((f_corrected[test_idx]  - f_fine_np[test_idx])**2)  / np.mean((f_coarse_np[test_idx]  - f_fine_np[test_idx])**2):.1%}")

    # ── Slab maps: |ΔF pred|, |error|, coloured by region ────────────────
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # |ΔF| ground truth
    hb = axes[0].hexbin(px, py, C=df_mag[slab_mask], gridsize=55,
                        cmap="Oranges", reduce_C_function=np.mean)
    plt.colorbar(hb, ax=axes[0]); axes[0].set_title("|ΔF| ground truth")

    # |ΔF pred|
    pred_mag = np.sqrt(np.sum(pred_all**2, axis=-1))
    hb = axes[1].hexbin(px, py, C=pred_mag[slab_mask], gridsize=55,
                        cmap="Blues", reduce_C_function=np.mean)
    plt.colorbar(hb, ax=axes[1]); axes[1].set_title("|ΔF pred|")

    # error
    hb = axes[2].hexbin(px, py, C=err_all[slab_mask], gridsize=55,
                        cmap="Reds", reduce_C_function=np.mean)
    plt.colorbar(hb, ax=axes[2]); axes[2].set_title("|pred - target|")

    # region label (0=train, 1=test)
    region_label = np.zeros(len(pos_np))
    region_label[test_idx] = 1.0
    hb = axes[3].hexbin(px, py, C=region_label[slab_mask], gridsize=55,
                        cmap="RdBu", reduce_C_function=np.mean, vmin=0, vmax=1)
    plt.colorbar(hb, ax=axes[3]); axes[3].set_title("Region (0=train, 1=test)")

    for ax in axes: ax.set_xlabel("x")
    plt.suptitle(f"Spatial generalisation maps  a={a:.3f}  (slab projection)", fontsize=11)
    plt.tight_layout(); plt.show()

    # ── Error CDF: train vs test ──────────────────────────────────────────
    thresholds = np.linspace(0, np.percentile(err_all, 99), 300)
    cdf_train = [np.mean(err_all[train_idx] <= t) for t in thresholds]
    cdf_test  = [np.mean(err_all[test_idx]  <= t) for t in thresholds]
    cdf_base  = [np.mean(df_mag <= t) for t in thresholds]   # baseline: |ΔF|

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(thresholds, cdf_train, "tomato",    lw=2, label="train region")
    ax.plot(thresholds, cdf_test,  "steelblue", lw=2, label="test  region")
    ax.set_xlabel("|prediction error|  [mesh units]")
    ax.set_ylabel("CDF")
    ax.set_title("Error CDF: train vs test  (right-shifted = worse)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## Section 6 — Generalisation across snapshots

In [ ]:
if PARAMS is not None:
    import pandas as pd
    rows = []
    for snap_id in SNAPS_VAL:
        pv, vv, av = load_single_snapshot(DATA_DIR, SIM_VAL, snap_id, N_PART, BOX_SIZE)
        fc_v, ff_v, df_v = jax.jit(partial(compute_force_pair, mesh_lr=MESH_LR, mesh_hr=MESH_HR))(pv)
        fv_v, dv_D = snapshot_features(pv, neighbor_idx, N_PART, USE_STRAIN, USE_INVS)
        vel_v = vv if USE_VEL else jnp.zeros_like(vv)
        pred_v = np.asarray(jax.device_get(
            jax.jit(lag_model.apply)(PARAMS, fv_v, vel_v, jnp.array(av))
        ))
        df_v_np = np.asarray(jax.device_get(df_v))
        fc_v_np = np.asarray(jax.device_get(fc_v))
        ff_v_np = np.asarray(jax.device_get(ff_v))

        def rmse_improv(idx):
            base = float(np.mean((fc_v_np[idx] - ff_v_np[idx])**2))
            corr = float(np.mean((fc_v_np[idx] + pred_v[idx] - ff_v_np[idx])**2))
            return 1 - corr / base

        r_tr = float(np.mean([pearsonr(pred_v[train_idx,c], df_v_np[train_idx,c])[0] for c in range(3)]))
        r_te = float(np.mean([pearsonr(pred_v[test_idx, c], df_v_np[test_idx, c])[0]  for c in range(3)]))
        rows.append({
            "snap": snap_id, "a": av,
            "train_R": r_tr, "test_R": r_te, "gap": r_tr - r_te,
            "train_MSE_improv%": rmse_improv(train_idx)*100,
            "test_MSE_improv%":  rmse_improv(test_idx)*100,
        })
        del pv, vv, fv_v, dv_D, df_v, fc_v, ff_v, pred_v

    df_snaps = pd.DataFrame(rows)
    print(df_snaps.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df_snaps["a"], df_snaps["train_R"],       "o-", c="tomato",    lw=2, label="train R")
    axes[0].plot(df_snaps["a"], df_snaps["test_R"],        "o-", c="steelblue", lw=2, label="test R")
    axes[0].fill_between(df_snaps["a"], df_snaps["test_R"], df_snaps["train_R"],
                         alpha=0.15, color="gray", label="generalisation gap")
    axes[0].set_xlabel("a"); axes[0].set_ylabel("Pearson R")
    axes[0].set_title("Train R vs Test R across snapshots"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(df_snaps["a"], df_snaps["train_MSE_improv%"], "o-", c="tomato",    lw=2, label="train")
    axes[1].plot(df_snaps["a"], df_snaps["test_MSE_improv%"],  "o-", c="steelblue", lw=2, label="test")
    axes[1].axhline(0, color="k", ls="--", lw=0.8)
    axes[1].set_xlabel("a"); axes[1].set_ylabel("MSE improvement %")
    axes[1].set_title("MSE improvement: train vs test across snapshots")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()